# Biblioteka Bokeh

Bokeh to biblioteka wyróżniająca się wysokim poziomem interaktywności. Umożliwia dostosowywanie wizualizacji w czasie rzeczywistym  dla użytkowników, którzy nie mają styczności z kodem.

In [ ]:
from bokeh.io import output_notebook
from bokeh.plotting import figure, show
from bokeh.models import LassoSelectTool, PolySelectTool, WheelZoomTool, UndoTool, ResetTool, RedoTool, SaveTool, HoverTool
from bokeh.layouts import gridplot
from bokeh.models import FactorRange, CrosshairTool

In [ ]:
import numpy as np
import pandas as pd
#import matplotlib.pyplot as plt
#import seaborn as sns

athlete_events = pd.read_csv('athlete_events.csv', sep=';')

--------------------

##### ⭐ Zadanie 1: 

Przygotuj wykres punktowy (`circle`) opierając się na danych, które sam wybierzesz w sensowny sposób. W rozwiązaniu zdefiniuj własny pasek narzędzi składający się z narzędzi:

- `LassoSelectTool` i `xPanTool` z kategorii narzędzi *Gestures* (*Pan/Drag tools*),
- `PolySelectTool` z kategorii narzędzi *Gestures* (*Click/Tap tools*),
- `WheelZoomTool` z kategorii narzędzi *Gestures* (*Scroll/Pinch tools*),
- `UndoTool`, `RedoTool`, `ResetTool` i `SaveTool` z kategorii narzędzi *Actions*,
- `HoverTool` z etykietami danych z kategorii narzędzi *Inspectors*.

Zapoznaj się z parametrem `toolbar_location` i dobierz dla niego nową wartość. Zapoznaj się z metodą `autohide` obiektu `toolbar` i dobierz dla niego taką wartość, żeby pasek narzędzi chował się po zjechaniu kursorem myszy z wizualizacji. Zadbaj o czytelność wykresu (tytuł wykresu i podpisy osi). 

In [ ]:
scatter_data = athlete_events.loc[
    (athlete_events['Sport'] == 'Taekwondo') &
    (athlete_events['Sex'] == 'F')
    , ['Weight', 'Height']
].drop_duplicates().dropna()

output_notebook()

hover = HoverTool(
    description='Najedź',
    tooltips=[
        ("Waga", "@x kg"),
        ("Wzrost", "@y cm")
    ]
)

p = figure(
    title='Waga i wzrost zawodniczek taekwondo',
    width=800,
    height=800,
    x_axis_label='Waga (kg)',
    y_axis_label='Wzrost (cm)',
    toolbar_location='below',
    tools=[LassoSelectTool(description='Zaznacz punkty (Lasso)'), WheelZoomTool(description='Przybliż'), ResetTool(description='Resetuj'), PolySelectTool(description='Zaznacz punkty (Wielokąt)'), UndoTool(description='Powrót'), RedoTool(description='Powtórz'), SaveTool(description='Zapisz'), hover],
)

p.circle(scatter_data['Weight'], scatter_data['Height'], size=15, color='navy', alpha=0.5)
p.toolbar.autohide = True

show(p)

##### ⭐ Zadanie 2:

Przygotuj wykres kolumnowy (`vbar`) opierając się na danych, które sam wybierzesz w sensowny sposób. Uwzględnij co najmniej 3 serie danych. Każda seria danych musi być przedstawiona na osobnym podwykresie (`subplot`) jednego obrazu. Zapoznaj się z metodami `column`, `row` i `gridplot` oraz dobierz dla nich właściwą wartość mając na uwadze wizualizację określonej liczby serii danych na osobnych podwykresach. Dodaj powiązane zachowania pomiędzy podwykresami:

- współdzielony ruchomy zakres dla wartości na osi X, także przy poziomym przesuwaniu wykresu,
- współdzielony wskaźnik będący narzędziem `CrosshairTool` widoczny jednocześnie na wszystkich podwykresach.

Zadbaj o czytelność wykresu (tytuł wykresu, podpisy osi i ewentualnie legenda). 

In [ ]:
swim = r"^Athletics Men's (100|400|1,500|10,000) metres"

vbar_data = athlete_events.loc[
    (athlete_events['Sport'] == 'Athletics') & 
    (athlete_events['Sex'] == 'M') & 
    (athlete_events['Event'].str.contains(swim, regex=True)) & 
    (athlete_events['Year'] >= 1980) & 
    (athlete_events['Medal'].notna()), 
    ['Year', 'Event', 'Weight']
].groupby(['Year', 'Event']).mean().reset_index()

vbar_data['Year'] = vbar_data['Year'].astype(str)

crosshair = CrosshairTool(
    dimensions='both',
    line_color='red',
    line_width=1,
    line_alpha=0.5
)

figures = [
    figure(
        title=f'Średnia waga medalistów w {event}',
        width=250,
        height=250,
        x_axis_label='Rok',
        y_axis_label='Wzrost (cm)',
        toolbar_location='below', 
        x_range=FactorRange(factors=vbar_data['Year'].unique()),
        tools=[crosshair]
    ) for event in vbar_data['Event'].unique()
]

for i, event in enumerate(vbar_data['Event'].unique()):
    tmp = vbar_data[vbar_data['Event'] == event]
    
    figures[i].vbar(
        x=tmp['Year'],
        top=tmp['Weight'],
        width=0.7,
        color="#1f77b4",
        legend_label=event
    )
    figures[i].title.text_color = "#333333"
    figures[i].background_fill_color = "#f5f5f5"
    figures[i].background_fill_alpha = 0.8
    figures[i].grid.grid_line_color = "#dddddd"
    figures[i].grid.grid_line_alpha = 0.7
    figures[i].legend.location = 'top_left'
    figures[i].legend.click_policy = 'hide'

grid = gridplot([[figures[2], figures[3]], [figures[0], figures[1]]], width=800, height=800)
show(grid)

##### ⭐ Zadanie 3:

Przygotuj wykres liniowy (`line`) opierając się na danych, które sam wybierzesz w sensowny sposób. Uwzględnij co najmniej 3 serie danych. Zapoznaj się z parametrem `location` obiektu `legend` i dobierz dla niego nową wartość. Zapoznaj się z parametrem `click_policy` obiektu `legend` i dobierz dla niego taką wartość, żeby po kliknięciu w tytuł danej serii danych w legendzie, jej wizualizacji znikała z wykresu. Zadbaj o czytelność wykresu (tytuł wykresu, podpisy osi i legenda). 

In [ ]:
swim = r"^Swimming Men's (100|400|1,500) metres Freestyle$"
line_data = athlete_events.loc[
    (athlete_events['Sport'] == 'Swimming') & 
    (athlete_events['Sex'] == 'M') & 
    (athlete_events['Event'].str.contains(swim, regex=True)) & 
    (athlete_events['Year'] >= 1952) & 
    (athlete_events['Medal'].notna()), 
    ['Year', 'Event', 'Height']
].groupby(['Year', 'Event']).mean().reset_index()

p = figure(
    title='Średni wzrost medalistów w pływaniu',
    width=800,
    height=800,
    x_axis_label='Rok',
    y_axis_label='Wzrost (cm)',
)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for i, event in enumerate(line_data['Event'].unique()):
    tmp = line_data[line_data['Event'] == event]
    event_name = event.split("'")[1]
    p.line(tmp['Year'], tmp['Height'], line_width=2, color=colors[i], legend_label=event_name)
    p.circle(tmp['Year'], tmp['Height'], size=6, color=colors[i], alpha=0.7)

p.legend.location = 'top_left'
p.legend.click_policy = 'hide'
p.title.text_color = "#333333"
p.background_fill_color = "#f5f5f5"
p.background_fill_alpha = 0.8
p.grid.grid_line_color = "#dddddd"
p.grid.grid_line_alpha = 0.7

output_notebook()
show(p)

##### ⭐ Zadanie 4:

Przedstaw na dowolnym wykresie dowolne zestawienie danych przygotowane na podstawie pliku `athlete_events.csv` z pierwszego tygodnia. Zadbaj o czytelność wykresu. Do swojej wizualizacji dodaj co najmniej 3 różne widgety, które będą miały na nią wpływ:

- `AutocompleteInput`,
- `Button`,
- `CheckboxButtonGroup`,
- `CheckboxGroup`,
- `ColorPicker`,
- `DataCube`,
- `DataTable`,
- `DatePicker`,
- `DateRangePicker`,
- `MultipleDatePicker`,
- `DatetimePicker`,
- `DatetimeRangePicker`,
- `MultipleDatetimePicker`,
- `TimePicker`,
- `DateRangeSlider`,
- `DateSlider`,
- `DatetimeRangeSlider`,
- `Div`,
- `Dropdown`,
- `FileInput`,
- `MultiChoice`,
- `MultiSelect`,
- `NumericInput`,
- `Paragraph`,
- `PasswordInput`,
- `PreText`,
- `RadioButtonGroup`,
- `RadioGroup`,
- `RangeSlider`,
- `Select`,
- `Slider`,
- `Spinner`,
- `Switch`,
- `Tabs`,
- `TextAreaInput`,
- `TextInput`,
- `Toggle`.

Dodatkowo, zastosuj `HelpButton` z `Tooltip` lub dodaj `Tooltip` do przynajmniej 1 widgetu.

In [ ]:
num_of_events_summer = athlete_events.loc[athlete_events.Season == 'Summer', ['Event', 'Year', 'City']].drop_duplicates().groupby(['Year', 'City']).count().reset_index()
num_of_events_summer.rename(columns={'Event': 'Count'}, inplace=True)
num_of_events_summer['Location'] = num_of_events_summer['Year'].astype(str) + ' ' + num_of_events_summer['City']
num_of_events_summer.drop(columns=['Year', 'City'], inplace=True)

num_of_events_summer

In [ ]:
p = figure(
    title='Liczba konkurencji na letnich igrzyskach',
    width=800,
    height=800,
    x_axis_label='Lokalizacja',
    y_axis_label='Liczba konkurencji',
)

p.line(
    num_of_events_summer['Location'],
    num_of_events_summer['Count'],
    line_width=2
)

output_notebook()
show(p)